# NTPC PPE Detection PoC: Dataset Preparation (Stage 2 Crops)

This notebook downloads the public **HardHat-Vest Dataset v3** from Kaggle, runs a pre-trained YOLOv11n model to extract person crops, maps the ground-truth annotations (`helmet`, `head`, `vest`) into coordinates relative to each person crop, and saves the new cropped dataset for training the Stage 2 model.

In [ ]:
# 1. Install required packages
!pip install -q kaggle opencv-python ultralytics tqdm

In [ ]:
# 2. Setup Kaggle API Credentials
# Place your kaggle.json in ~/.kaggle/ first, then run this block
import os
from pathlib import Path

kaggle_dir = Path.home() / ".kaggle"
if not (kaggle_dir / "kaggle.json").exists():
    print("WARNING: kaggle.json not found in ~/.kaggle/. Please download your API key and place it there.")
else:
    print("Kaggle API credentials found!")

In [ ]:
# 3. Download the Dataset
!kaggle datasets download -d muhammetzahitaydn/hardhat-vest-dataset-v3 --unzip -p ./raw_dataset

In [ ]:
# 4. Define Person Cropping and Label Mapping Logic
import cv2
import numpy as np
from tqdm import tqdm
from ultralytics import YOLO
import shutil
from pathlib import Path

# Load standard COCO YOLO11n for person detection
# Using just the model name lets Ultralytics auto-download if not found
person_model = YOLO('yolo11n.pt')

out_dir = Path('../datasets/ppe_crops')
out_dir.mkdir(parents=True, exist_ok=True)

# Create standard YOLO dataset directories
for split in ['train', 'val']:
    (out_dir / split / 'images').mkdir(parents=True, exist_ok=True)
    (out_dir / split / 'labels').mkdir(parents=True, exist_ok=True)

def box_containment_ratio(box1, box2):
    """Return the fraction of box1 area that lies inside box2.
    Box format: [x1, y1, x2, y2] (absolute pixel coords).
    """
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    if x2 < x1 or y2 < y1: return 0.0
    inter = (x2 - x1) * (y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    if area1 <= 0: return 0.0
    return inter / area1

In [ ]:
# 5. Process Raw Images and Labels
def process_split(split_name, src_images_dir, src_labels_dir):
    # Validate directories exist before processing
    for d in [src_images_dir, src_labels_dir]:
        p = Path(d)
        if not p.exists():
            raw = Path('./raw_dataset')
            print(f'ERROR: Expected directory not found: {p.resolve()}')
            if raw.exists():
                print(f'  Contents of raw_dataset: {[x.name for x in raw.iterdir()]}')
            raise FileNotFoundError(f'Missing: {d}')

    img_paths = list(Path(src_images_dir).glob('*.jpg')) + list(Path(src_images_dir).glob('*.png'))
    crop_count = 0

    for img_path in tqdm(img_paths, desc=f'Processing {split_name}'):
        img = cv2.imread(str(img_path))
        if img is None: continue
        h_img, w_img = img.shape[:2]

        # Load ground truth labels (format: class_id x_center y_center width height)
        label_path = Path(src_labels_dir) / f'{img_path.stem}.txt'
        if not label_path.exists(): continue

        gt_boxes = []
        with open(label_path, 'r') as f:
            for line in f.read().strip().split('\n'):
                if not line: continue
                parts = line.split()
                cid = int(parts[0])
                xc, yc, w, h = map(float, parts[1:])
                x1 = (xc - w/2) * w_img
                y1 = (yc - h/2) * h_img
                x2 = (xc + w/2) * w_img
                y2 = (yc + h/2) * h_img
                gt_boxes.append((cid, [x1, y1, x2, y2]))

        # Run person detector (Stage 1)
        results = person_model(img, verbose=False)[0]

        person_boxes = []
        for box in results.boxes:
            if int(box.cls[0]) == 0 and float(box.conf[0]) > 0.50:
                px1, py1, px2, py2 = map(int, box.xyxy[0].tolist())
                person_boxes.append([px1, py1, px2, py2])

        for idx, p_box in enumerate(person_boxes):
            px1, py1, px2, py2 = p_box
            pw = px2 - px1
            ph = py2 - py1
            if pw <= 0 or ph <= 0: continue

            person_crop = img[py1:py2, px1:px2]

            crop_labels = []
            for cid, gt_box in gt_boxes:
                overlap = box_containment_ratio(gt_box, p_box)
                if overlap > 0.50:
                    rx1 = max(0, gt_box[0] - px1)
                    ry1 = max(0, gt_box[1] - py1)
                    rx2 = min(pw, gt_box[2] - px1)
                    ry2 = min(ph, gt_box[3] - py1)
                    r_xc = ((rx1 + rx2) / 2.0) / pw
                    r_yc = ((ry1 + ry2) / 2.0) / ph
                    r_w = (rx2 - rx1) / pw
                    r_h = (ry2 - ry1) / ph
                    crop_labels.append(f'{cid} {r_xc:.6f} {r_yc:.6f} {r_w:.6f} {r_h:.6f}')

            if crop_labels:
                crop_name = f'{img_path.stem}_p{idx}'
                cv2.imwrite(str(out_dir / split_name / 'images' / f'{crop_name}.jpg'), person_crop)
                with open(out_dir / split_name / 'labels' / f'{crop_name}.txt', 'w') as f:
                    f.write('\n'.join(crop_labels))
                crop_count += 1

    print(f'Created {crop_count} person crops for {split_name} split.')

process_split('train', './raw_dataset/images/train', './raw_dataset/labels/train')
process_split('val', './raw_dataset/images/val', './raw_dataset/labels/val')

In [ ]:
# 6. Generate data.yaml
yaml_content = f'''
path: {out_dir.resolve()}
train: train/images
val: val/images

nc: 3
names: ['helmet', 'head', 'vest']
'''
with open('../datasets/ppe_crops/data.yaml', 'w') as f:
    f.write(yaml_content.strip())
print('data.yaml generated!')
print(f'Location: {Path("../datasets/ppe_crops/data.yaml").resolve()}')